# 03 - Referencia avanzada: publicar la base analitica

Lee primero `docs/modulos/03-base-analitica.md` y `docs/contrato-analitico.md`.
Ejecuta el notebook 01 y adjunta aqui el mismo Lakehouse dedicado al taller.
No necesitas ejecutar la solucion 02: este notebook incorpora sus funciones.

**Por defecto no escribe.** Revisa el Lakehouse y SCHEMA; ejecuta hasta la validacion,
compara con el informe del notebook 01 y solo despues autoriza WRITE_ANALYTICS.
Los destinos son workshop_sales_clean, workshop_sales_daily y workshop_calendar.
Conserva las tres tablas originales; nunca sobrescribe tablas derivadas existentes.
Las escrituras no son atomicas: ante un fallo puede quedar una publicacion parcial.
No reintentes ni borres automaticamente; revisa el estado antes de decidir.


In [ ]:
SCHEMA = None  # Confirma "dbo", otro esquema existente o "" sin esquemas.
WRITE_ANALYTICS = False


In [ ]:
"""Fabric notebook helpers. Uses the existing notebook Spark session."""

import re

TABLE_NAMES = {
    "products": "workshop_products",
    "stores": "workshop_stores",
    "sales_raw": "workshop_sales_raw",
}


def table_names(schema):
    if not isinstance(schema, str) or (
        schema and not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema)
    ):
        raise ValueError("SCHEMA debe ser un identificador simple o una cadena vacia.")
    prefix = f"`{schema}`." if schema else ""
    return {key: f"{prefix}`{value}`" for key, value in TABLE_NAMES.items()}


def require_default_lakehouse(context):
    if not context.get("defaultLakehouseId"):
        raise RuntimeError(
            "Adjunta un Lakehouse como predeterminado al notebook y reinicia la sesion Spark."
        )


def create_dataframes(spark, data):
    from pyspark.sql import types as T

    schemas = {
        "products": T.StructType(
            [
                T.StructField("product_id", T.IntegerType(), False),
                T.StructField("product_name", T.StringType(), False),
                T.StructField("category", T.StringType(), False),
                T.StructField("list_price", T.DecimalType(18, 2), False),
            ]
        ),
        "stores": T.StructType(
            [
                T.StructField("store_id", T.IntegerType(), False),
                T.StructField("store_name", T.StringType(), False),
                T.StructField("city", T.StringType(), False),
            ]
        ),
        "sales_raw": T.StructType(
            [
                T.StructField("sale_id", T.StringType(), False),
                T.StructField("sale_date", T.DateType(), False),
                T.StructField("product_id", T.IntegerType(), False),
                T.StructField("store_id", T.IntegerType(), False),
                T.StructField("channel", T.StringType(), False),
                T.StructField("quantity", T.IntegerType(), False),
                T.StructField("unit_price", T.DecimalType(18, 2), False),
                T.StructField("discount_rate", T.DecimalType(4, 2), False),
                T.StructField("status", T.StringType(), False),
                T.StructField("category_raw", T.StringType(), True),
                T.StructField("city_raw", T.StringType(), True),
            ]
        ),
    }
    return {key: spark.createDataFrame(data[key], schema) for key, schema in schemas.items()}


def write_tables(spark, frames, schema, allow_overwrite=False):
    if type(allow_overwrite) is not bool:
        raise ValueError("ALLOW_OVERWRITE debe ser True o False.")
    names = table_names(schema)
    if set(frames) != set(names):
        raise ValueError("Se requieren exactamente products, stores y sales_raw.")
    if schema:
        spark.sql(f"SHOW TABLES IN `{schema}`").limit(1).collect()
    existing = [name for name in names.values() if spark.catalog.tableExists(name)]
    if existing and not allow_overwrite:
        raise FileExistsError(
            "No se ha escrito ninguna tabla. Ya existen: "
            + ", ".join(existing)
            + ". Usa otro Lakehouse o revisa ALLOW_OVERWRITE."
        )
    mode = "overwrite" if allow_overwrite else "errorifexists"
    for key, name in names.items():
        writer = frames[key].write.format("delta").mode(mode)
        if allow_overwrite:
            writer = writer.option("overwriteSchema", "true")
        writer.saveAsTable(name)
    counts = {key: spark.table(name).count() for key, name in names.items()}
    for key, actual in counts.items():
        expected = frames[key].count()
        if actual != expected:
            raise RuntimeError(f"Recuento incorrecto en {names[key]}: {actual} != {expected}")
    return counts


In [ ]:
"""Reference solution: exact deduplication, canonical dimensions and decimal revenue."""


def clean_sales(sales_raw, products, stores):
    from pyspark.sql import functions as F

    for frame, key in ((products, "product_id"), (stores, "store_id")):
        if frame.filter(F.col(key).isNull()).limit(1).count():
            raise ValueError(f"La dimension contiene una clave nula: {key}")
        if frame.groupBy(key).count().filter(F.col("count") > 1).limit(1).count():
            raise ValueError(f"La dimension contiene claves duplicadas: {key}")

    deduplicated = sales_raw.dropDuplicates()
    if deduplicated.filter(F.col("sale_id").isNull()).limit(1).count():
        raise ValueError("Hay sale_id nulos. Revisa los datos antes de continuar.")
    if deduplicated.groupBy("sale_id").count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError("Hay sale_id repetidos con contenido diferente. No se elige uno al azar.")

    eligible = deduplicated.filter((F.col("quantity") > 0) & (F.col("status") == "completed"))
    enriched = eligible.join(products.select("product_id", "category"), "product_id", "inner").join(
        stores.select("store_id", "city"), "store_id", "inner"
    )
    return enriched.select(
        "sale_id",
        "sale_date",
        "product_id",
        "store_id",
        "channel",
        "quantity",
        "unit_price",
        "discount_rate",
        "status",
        "category",
        "city",
        F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1) - F.col("discount_rate")), 2)
        .cast("decimal(18,2)")
        .alias("net_amount"),
    )


In [ ]:
"""Optional analytics outputs, separate from the unchanged raw-data generator."""

from datetime import date
from re import fullmatch

ANALYTICS_TABLE_NAMES = {
    "sales_clean": "workshop_sales_clean",
    "sales_daily": "workshop_sales_daily",
    "calendar": "workshop_calendar",
}


def analytics_table_names(schema):
    if not isinstance(schema, str) or (schema and not fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema)):
        raise ValueError("SCHEMA debe ser un identificador simple o una cadena vacia.")
    prefix = f"`{schema}`." if schema else ""
    return {key: f"{prefix}`{value}`" for key, value in ANALYTICS_TABLE_NAMES.items()}


def build_analytics_frames(sales_clean):
    from pyspark.sql import functions as F

    required = (
        "sale_id",
        "sale_date",
        "product_id",
        "store_id",
        "channel",
        "quantity",
        "net_amount",
        "category",
        "city",
    )
    invalid = F.col("status").isNull() | (F.col("status") != "completed")
    for column in required:
        invalid = invalid | F.col(column).isNull()
    invalid = invalid | (F.col("quantity") <= 0)
    if sales_clean.filter(invalid).limit(1).count():
        raise ValueError("La entrada limpia contiene nulos o lineas no elegibles.")
    if sales_clean.groupBy("sale_id").count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError("La entrada limpia contiene sale_id repetidos.")

    bounds = sales_clean.agg(
        F.min("sale_date").alias("start"), F.max("sale_date").alias("end")
    ).first()
    if bounds.start is None:
        raise ValueError("No hay ventas limpias; no se publican tablas vacias.")
    calendar = (
        sales_clean.sparkSession.range(1)
        .select(
            F.explode(
                F.sequence(
                    F.lit(date(bounds.start.year, 1, 1)),
                    F.lit(date(bounds.end.year, 12, 31)),
                )
            ).alias("calendar_date")
        )
        .select(
            "calendar_date",
            F.year("calendar_date").alias("calendar_year"),
            F.quarter("calendar_date").alias("calendar_quarter"),
            F.month("calendar_date").alias("month_number"),
            F.date_format("calendar_date", "yyyy-MM").alias("year_month"),
        )
    )
    daily = sales_clean.groupBy("sale_date", "product_id", "store_id", "channel").agg(
        F.count("*").alias("sales_count"),
        F.sum("quantity").alias("units"),
        F.sum("net_amount").cast("decimal(38,2)").alias("net_revenue"),
    )
    return {"sales_clean": sales_clean, "sales_daily": daily, "calendar": calendar}


def validate_analytics_frames(frames):
    from pyspark.sql import functions as F

    clean = (
        frames["sales_clean"]
        .agg(
            F.count("*").alias("lines"),
            F.sum("quantity").alias("units"),
            F.sum("net_amount").alias("revenue"),
        )
        .first()
    )
    daily = (
        frames["sales_daily"]
        .agg(
            F.sum("sales_count").alias("lines"),
            F.sum("units").alias("units"),
            F.sum("net_revenue").alias("revenue"),
        )
        .first()
    )
    if tuple(clean) != tuple(daily):
        raise ValueError("El agregado diario no reconcilia con el detalle limpio.")
    return {"clean_rows": clean.lines, "units": clean.units, "net_revenue": clean.revenue}


def write_analytics_tables(spark, frames, schema, approved=False):
    if approved is not True:
        raise PermissionError("Publicacion desactivada: revisa y autoriza WRITE_ANALYTICS.")
    names = analytics_table_names(schema)
    if set(frames) != set(names):
        raise ValueError("Se requieren exactamente sales_clean, sales_daily y calendar.")
    if schema:
        spark.sql(f"SHOW TABLES IN `{schema}`").limit(1).count()
    existing = [name for name in names.values() if spark.catalog.tableExists(name)]
    if existing:
        raise FileExistsError(
            "No se ha escrito ninguna tabla. Ya existen: "
            + ", ".join(existing)
            + ". No se permite sobrescribir; revisa el destino."
        )
    expected = validate_analytics_frames(frames)
    expected_counts = {key: frame.count() for key, frame in frames.items()}
    for key, name in names.items():
        frames[key].write.format("delta").mode("errorifexists").saveAsTable(name)
    saved = {key: spark.table(name) for key, name in names.items()}
    for key, frame in saved.items():
        if frame.count() != expected_counts[key]:
            raise RuntimeError(f"Recuento incorrecto tras publicar {names[key]}.")
    if validate_analytics_frames(saved) != expected:
        raise RuntimeError("Los totales publicados no coinciden con la entrada.")
    return expected


In [ ]:
import notebookutils

require_default_lakehouse(notebookutils.runtime.context)
source_lakehouse_id = notebookutils.runtime.context["defaultLakehouseId"]
names = table_names(SCHEMA)
targets = analytics_table_names(SCHEMA)
print("Origenes:", names)
print("Destinos propuestos (no escritos):", targets)
source_frames = {key: spark.table(name) for key, name in names.items()}
expected_frames = create_dataframes(spark, {key: [] for key in names})
for key, frame in source_frames.items():
    frame.printSchema()
    actual = {field.name: field.dataType for field in frame.schema.fields}
    expected = {
        field.name: field.dataType for field in expected_frames[key].schema.fields
    }
    if actual != expected:
        raise ValueError(f"Esquema distinto del contrato: {names[key]}")


In [ ]:
sales_clean = clean_sales(
    source_frames["sales_raw"], source_frames["products"], source_frames["stores"]
)
analytics_frames = build_analytics_frames(sales_clean)
totals = validate_analytics_frames(analytics_frames)
print("Totales calculados en esta ejecucion:", totals)
for key, frame in analytics_frames.items():
    print(key, frame.count())
    frame.printSchema()
display(analytics_frames["sales_daily"].orderBy(
    "sale_date", "product_id", "store_id", "channel"
).limit(20))


## Puerta de escritura

Detente y compara los totales con el informe del generador. Confirma los destinos
con quien administra el taller. Cambiar el parametro no concede permisos Fabric.
Si autorizas, cambia WRITE_ANALYTICS a True, reejecuta solo parametros y esta celda.
No cambies SCHEMA entre la inspeccion y la publicacion.


In [ ]:
if type(WRITE_ANALYTICS) is not bool:
    raise ValueError("WRITE_ANALYTICS debe ser True o False.")
if notebookutils.runtime.context.get("defaultLakehouseId") != source_lakehouse_id:
    raise ValueError("El Lakehouse ha cambiado. Repite inspeccion y validacion.")
if analytics_table_names(SCHEMA) != targets:
    raise ValueError("El destino ha cambiado. Repite inspeccion y validacion.")
if WRITE_ANALYTICS:
    published = write_analytics_tables(
        spark, analytics_frames, SCHEMA, approved=WRITE_ANALYTICS
    )
    print("Tablas guardadas y reconciliadas:", published)
else:
    print("Solo calculo en memoria. No se ha escrito ninguna tabla.")


## Reejecucion y siguientes modulos

Devuelve WRITE_ANALYTICS a False al terminar. Una segunda publicacion se detiene si
existe cualquiera de los tres destinos, antes de escribir ninguno. Ante colisiones
usa un entorno de taller nuevo; no desactives la proteccion.

Solo tras publicar y validar, continua con modelo e informe, ontologia, Data Agent
o Native App. No cargues simultaneamente detalle y agregado como hechos sumables
en el modelo: duplicarias los ingresos. El modelo de referencia usa el detalle.
